# Atelier Scikit-learn

Une entreprise possède plusieurs bâtiments équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression, la consommation énergétique, le bâtiment, la date et l'heure de la mesure.  

Chaque mesure possède également un état (OK, ALERTE et ERREUR). L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un capteur à partir de ses mesures. 

L'atelier suivra le workflow classique du Machine Learning : Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation 

## Partie 0 – mise en place de l’environnement 

**Objectif** : préparer l'environnement de travail en important les librairies nécessaires (manipulation de données et visualisation), avant de charger et explorer le dataset.

In [ ]:
#Installer et importer seaborn, matplotlib et pandas si pas encore
#%pip install pandas seaborn matplotlib scikit-learn

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Pour un affichage plus agréable des graphiques dans le notebook
%matplotlib inline

### Importer mesures_capteurs.csv dans le dataframe df et Explorer le dataframe df 

In [2]:
df = pd.read_csv("../data/mesures_capteurs.csv")

**Explication** : `pd.read_csv()` lit le fichier CSV situé dans `data/` et le charge dans un DataFrame Pandas nommé `df`. Le chemin `../data/mesures_capteurs.csv` remonte d'un niveau (depuis `notebooks/`) puis entre dans `data/`, ce qui suppose que le notebook est exécuté depuis le dossier `notebooks/`.

**Résultat** : le dataset est chargé en mémoire sous forme de tableau, prêt à être exploré.

In [3]:
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


**Explication** : `df.head()` affiche par défaut les 5 premières lignes du DataFrame, pour avoir un premier aperçu visuel des données.

**Résultat** : on voit les colonnes `id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat`, avec des valeurs cohérentes (ex. température autour de 20-30°C, état "OK").

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


**Explication** : `df.info()` affiche pour chaque colonne son type (`object` pour du texte, `float64` pour des nombres décimaux) ainsi que le nombre de valeurs non nulles. Cela permet de repérer immédiatement les valeurs manquantes par colonne.

**Résultat** : le dataset contient 605 lignes. Les colonnes `temperature`, `humidite`, `pression`, `consommation` et `etat` contiennent chacune quelques valeurs manquantes (moins de 600 valeurs non nulles sur 605).

In [5]:
df.describe()

,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


In [6]:
df.shape

(605, 9)

**Explication** : `df.shape` retourne un tuple (nombre de lignes, nombre de colonnes).

**Résultat** : le dataset contient **605 lignes** et **9 colonnes**.

In [7]:
df.columns

Index(['id_mesure', 'date_heure', 'id_capteur', 'batiment', 'temperature',
       'humidite', 'pression', 'consommation', 'etat'],
      dtype='str')

**Explication** : `df.columns` liste les noms de toutes les colonnes du DataFrame.

**Résultat** : `id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat`.

In [8]:
df["etat"].value_counts(dropna=False)

etat
OK        567
ALERTE     29
ERREUR      5
NaN         4
Name: count, dtype: int64

**Explication** : `value_counts()` compte le nombre d'occurrences de chaque valeur unique de la colonne `etat` (la cible qu'on cherche à prédire). Le paramètre `dropna=False` inclut aussi le comptage des valeurs manquantes (`NaN`).

**Résultat** : la colonne `etat` contient 567 mesures "OK", 29 "ALERTE", 5 "ERREUR", et 4 valeurs manquantes.

## Partie 1 – Gestion des doublons 

**Objectif** : vérifier si le dataset contient des lignes strictement identiques (doublons), qui pourraient fausser l'entraînement du modèle en sur-représentant artificiellement certaines observations, puis les supprimer le cas échéant.

### 1) vérifier l’existence de doublons dans df 

In [9]:
print("Nombre de doublons :", df.duplicated().sum())
df[df.duplicated()]

Nombre de doublons : 5


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


**Explication** : `df.duplicated()` retourne une série de booléens (`True` si la ligne est identique à une ligne déjà rencontrée précédemment, `False` sinon). `.sum()` additionne les `True` (comptés comme 1), donnant ainsi le nombre total de doublons. `df[df.duplicated()]` permet d'afficher ces lignes dupliquées.

**Résultat** : le dataset contient **5 doublons**.

### 2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [10]:
print("Shape avant suppression :", df.shape)

df = df.drop_duplicates()

print("Shape après suppression :", df.shape)
print("Doublons restants :", df.duplicated().sum())

Shape avant suppression : (605, 9)
Shape après suppression : (600, 9)
Doublons restants : 0


**Explication** : `df.drop_duplicates()` supprime les lignes dupliquées en ne conservant que la première occurrence de chaque ligne identique. On revérifie ensuite avec `duplicated().sum()` pour confirmer qu'il n'en reste plus.

**Résultat** : le dataset passe de **605 à 600 lignes** après suppression des 5 doublons. La vérification confirme **0 doublon restant**.